# Centaur IME: 言語ごとの小さな模型の学習（GPU）

使い方: 「ランタイム → ランタイムのタイプを変更」で **A100 GPU** を選び（Colab Pro。T4 は無料でも使えるので選ばない）、「ランタイム → すべてのセルを実行」。

**最初のセルで Google ドライブにつなぐ**（許可を求められたら許可する。飛ばさない）。模型と結果は 1 つ終わるごとに
ドライブ（CentaurIME/models/2026-11-07）に直接書く。結果の数字は各セルの出力にも出る。

学習する模型（大事な順）:
1. 中国語: 声調なしの拼音 → 漢字（比べる相手: Rime で字の誤り 12.94%）
2. ベトナム語: 記号なし → 記号つき（比べる相手: 文 66.2%）
3. ヒンディー語: ローマ字 → デーヴァナーガリー（比べる相手: 語 68.5%）
4. 韓国語: 分かち書き（比べる相手: 文 68.8%、kiwipiepy 77.3%）
5. アラビア語（モロッコ方言）: Arabizi → アラビア文字（比べる相手: 語 70.2%）

材料は台帳（tools/data/sources.tsv）の決まった版を、この Colab の中にだけ落として使う。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 最初に Google ドライブにつなぐ（模型と結果は 1 つ終わるごとにドライブに直接書く。Colab が切れても消えない）
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CentaurIME/models/2026-11-07'
import os; os.makedirs(OUT, exist_ok=True); print('保存先', OUT)

In [ ]:
!rm -rf repo && git clone -q --depth 1 -b work/lang-measure https://github.com/J-KAMMA/centaur-ime.git repo && git -C repo log -1 --format='%H %s'
!pip -q install pypinyin==0.53.0 "opencc>=1.1,<2" pyarrow==21.0.0

In [ ]:
# 材料（決まった版。SHA-256 を記録に出す）
!python repo/tools/fetch_source.py tatoeba-cmn data/zh
!python repo/tools/fetch_source.py wiki-zh data/zh
!python repo/tools/fetch_source.py cc100-zh-Hans data/zh --lines 400000
!python repo/tools/fetch_source.py tatoeba-vie data/vi
!python repo/tools/fetch_source.py cc100-vi data/vi --lines 400000
!python repo/tools/fetch_source.py dakshina data/hi --member 'dakshina_dataset_v1.0/hi/*'
!python repo/tools/fetch_source.py tatoeba-kor data/ko
!python repo/tools/fetch_source.py wiki-ko data/ko
!python repo/tools/fetch_source.py cc100-ko data/ko --lines 300000
!python repo/tools/fetch_source.py arabizi-darija data/darija
# 中国語のウィキを 1 行 1 文にする
import re, pyarrow.parquet as pq, glob
t = pq.read_table(glob.glob('data/zh/*.parquet')[0], columns=['text'])
n = 0
with open('data/zh/wiki.txt', 'w', encoding='utf-8') as f:
    for text in t.column('text').to_pylist():
        for s in re.split(r'[。！？\n]', text):
            if 4 <= len(s) <= 60:
                f.write(s + '\n'); n += 1
        if n >= 400000: break
print('ウィキの文', n)

In [ ]:
!python repo/tools/models/train_task.py --task zh --tatoeba data/zh/cmn_sentences.tsv.bz2 --wiki-text data/zh/wiki.txt --cc100 data/zh/cc100-zh-Hans.400000.txt --extra 400000 --d 512 --layers 8 --heads 8 --batch 256 --minutes 90 --oversample 5 --amp --out "$OUT"
!ls -la "$OUT" | grep zh

In [ ]:
!python repo/tools/models/train_task.py --task vi --tatoeba data/vi/vie_sentences.tsv.bz2 --cc100 data/vi/cc100-vi.400000.txt --extra 400000 --d 384 --layers 8 --heads 6 --batch 256 --minutes 45 --amp --out "$OUT"
!ls -la "$OUT" | grep vi

In [ ]:
!python repo/tools/models/train_task.py --task hi --dakshina data/hi/dakshina_dataset_v1.0/hi --extra 60000 --d 384 --layers 8 --heads 6 --batch 256 --minutes 45 --amp --out "$OUT"
!ls -la "$OUT" | grep hi

In [ ]:
!python repo/tools/models/train_task.py --task ko --tatoeba data/ko/kor_sentences.tsv.bz2 --wiki data/ko/train-00002-of-00003.parquet --cc100 data/ko/cc100-ko.300000.txt --extra 300000 --d 384 --layers 8 --heads 6 --batch 256 --minutes 45 --amp --out "$OUT"
!ls -la "$OUT" | grep ko

In [ ]:
!python repo/tools/models/train_task.py --task ar --darija data/darija --d 256 --layers 6 --heads 4 --batch 64 --minutes 20 --amp --out "$OUT"
!ls -la "$OUT" | grep ar

In [ ]:
# 結果（出力のファイルで確かめる）
import glob
for f in sorted(glob.glob(OUT + '/*-result.txt')):
    print('==', f, os.path.getsize(f), 'バイト'); print(open(f, encoding='utf-8').read())
for f in sorted(glob.glob(OUT + '/*.pt')):
    print(f, round(os.path.getsize(f) / 1e6, 1), 'MB')